In [1]:
import torch
import numpy as np
import pandas as pd
from model import Net
model = [
    1131796,
    18094361,
    26317076,
    26318864,
    157564161,
    218105633,
    448585456,
    539035672,
    609641183,
    1211200801,
]
# 加载模型
model_list = []
for path in model:
    net = Net()
    print(f'Loading model from ../Training/model/model_DGM_fine_{path}.pth')
    net.load_state_dict(torch.load(f'../Training/model/model_DGM_fine_{path}.pth', weights_only=True, map_location=torch.device('cpu')))
    net.eval()
    net = net.float()
    net = torch.compile(net)
    model_list.append(net)
        
temp_fea = pd.read_csv('./TestSet.csv')
df_density_gas_keep = pd.read_csv('./Result/Gas_Compare.csv')[['Name', 'T(K)', 'P(Pa)', 'rho_GEoS']]
df_mean_std = pd.read_excel('../Database/feature_statistics.xlsx')
mean_ = df_mean_std['mean'].values
std_ = df_mean_std['std'].values

temp_fea = pd.merge(df_density_gas_keep, temp_fea, on='Name', how='left')
feature_tensor = torch.tensor((temp_fea.iloc[:, -31:].astype(np.double).values - mean_) / std_, dtype=torch.float32)
T = torch.tensor(temp_fea['T(K)'].values, dtype=torch.float32).view(-1, 1)
P = torch.tensor(temp_fea['P(Pa)'].values, dtype=torch.float32, requires_grad=True).view(-1, 1)
tc_vec = torch.tensor(temp_fea['critical temperature (K)'].values, dtype=torch.float32).view(-1, 1)
pc_vec = torch.tensor(temp_fea['critical pressure (Pa)'].values, dtype=torch.float32).view(-1, 1)
mw = temp_fea['molecular weight (unitless)'].values

T_data = torch.tensor(df_density_gas_keep['T(K)'].values, dtype=torch.float32).view(-1, 1)
P_data = torch.tensor(df_density_gas_keep['P(Pa)'].values, dtype=torch.float32).view(-1, 1)

chunk_size = 1024*4  # 根据显存调整
# chunk_size = T_data.shape[0]
v_list = []
n_samples = T_data.size(0)

for start in range(0, n_samples, chunk_size):
    end = min(start + chunk_size, n_samples)
    
    T_chunk = T_data[start:end].clone()
    P_chunk = P_data[start:end].clone().requires_grad_(True)
    
    Tr_chunk = T_chunk / tc_vec[start:end]
    Pr_chunk = P_chunk / pc_vec[start:end]
    fea_chunk = feature_tensor[start:end]
    
    phi_preds = []
    
    for net in model_list:
        phi_preds.append(net.forward_phi_only(fea_chunk, Tr_chunk, Pr_chunk))
    phi_res = torch.mean(torch.stack(phi_preds), dim=0)
    
    fug_final = 8.314 * T_chunk * (torch.log(P_chunk) + phi_res)
    grad_v = torch.autograd.grad(
        outputs=fug_final.sum(), 
        inputs=P_chunk, 
    )[0]
    
    # 转换并存储结果
    v_list.append(grad_v.detach().cpu().numpy().reshape(-1))
V = np.concatenate(v_list)

rho = 1e-3 * mw / V 
df_density_gas_keep['rho_pred'] = rho
# 检查是否与rho_GEoS相等
flag = False
for i in range(len(df_density_gas_keep)):
    if abs(df_density_gas_keep.loc[i, 'rho_pred'] - df_density_gas_keep.loc[i, 'rho_GEoS']) > 0.1:
        print(f"Difference at index {i}: rho_pred = {df_density_gas_keep.loc[i, 'rho_pred']}, rho_GEoS = {df_density_gas_keep.loc[i, 'rho_GEoS']}")
        flag = True
if not flag:
    print("All values are equal within the tolerance.")
else:
    print("There are differences between rho_pred and rho_GEoS (Gas).")

Loading model from ../Training/model/model_DGM_fine_1131796.pth
Loading model from ../Training/model/model_DGM_fine_18094361.pth
Loading model from ../Training/model/model_DGM_fine_26317076.pth
Loading model from ../Training/model/model_DGM_fine_26318864.pth
Loading model from ../Training/model/model_DGM_fine_157564161.pth
Loading model from ../Training/model/model_DGM_fine_218105633.pth
Loading model from ../Training/model/model_DGM_fine_448585456.pth
Loading model from ../Training/model/model_DGM_fine_539035672.pth
Loading model from ../Training/model/model_DGM_fine_609641183.pth
Loading model from ../Training/model/model_DGM_fine_1211200801.pth
All values are equal within the tolerance.


In [2]:
import torch
import numpy as np
import pandas as pd
from model import Net
model = [
    1131796,
    18094361,
    26317076,
    26318864,
    157564161,
    218105633,
    448585456,
    539035672,
    609641183,
    1211200801,
]
# 加载模型
model_list = []
for path in model:
    net = Net()
    print(f'Loading model from ../Training/model/model_DGM_fine_{path}.pth')
    net.load_state_dict(torch.load(f'../Training/model/model_DGM_fine_{path}.pth', weights_only=True, map_location=torch.device('cpu')))
    net.eval()
    net = net.float()
    net = torch.compile(net)
    model_list.append(net)
        
temp_fea = pd.read_csv('./TestSet.csv')
df_density_gas_keep = pd.read_csv('./Result/Liquid_Compare.csv')[['Name', 'T(K)', 'P(Pa)', 'rho_GEoS']]
df_mean_std = pd.read_excel('../Database/feature_statistics.xlsx')
mean_ = df_mean_std['mean'].values
std_ = df_mean_std['std'].values

temp_fea = pd.merge(df_density_gas_keep, temp_fea, on='Name', how='left')
feature_tensor = torch.tensor((temp_fea.iloc[:, -31:].astype(np.double).values - mean_) / std_, dtype=torch.float32)
T = torch.tensor(temp_fea['T(K)'].values, dtype=torch.float32).view(-1, 1)
P = torch.tensor(temp_fea['P(Pa)'].values, dtype=torch.float32, requires_grad=True).view(-1, 1)
tc_vec = torch.tensor(temp_fea['critical temperature (K)'].values, dtype=torch.float32).view(-1, 1)
pc_vec = torch.tensor(temp_fea['critical pressure (Pa)'].values, dtype=torch.float32).view(-1, 1)
mw = temp_fea['molecular weight (unitless)'].values

T_data = torch.tensor(df_density_gas_keep['T(K)'].values, dtype=torch.float32).view(-1, 1)
P_data = torch.tensor(df_density_gas_keep['P(Pa)'].values, dtype=torch.float32).view(-1, 1)

chunk_size = 1024*4  # 根据显存调整
# chunk_size = T_data.shape[0]
v_list = []
n_samples = T_data.size(0)

for start in range(0, n_samples, chunk_size):
    end = min(start + chunk_size, n_samples)
    
    T_chunk = T_data[start:end].clone()
    P_chunk = P_data[start:end].clone().requires_grad_(True)
    
    Tr_chunk = T_chunk / tc_vec[start:end]
    Pr_chunk = P_chunk / pc_vec[start:end]
    fea_chunk = feature_tensor[start:end]
    
    phi_preds = []
    
    for net in model_list:
        phi_preds.append(net.forward_background(fea_chunk, Tr_chunk, Pr_chunk))
    phi_res = torch.mean(torch.stack(phi_preds), dim=0)
    
    fug_final = 8.314 * T_chunk * (torch.log(P_chunk) + phi_res)
    grad_v = torch.autograd.grad(
        outputs=fug_final.sum(), 
        inputs=P_chunk, 
    )[0]
    
    # 转换并存储结果
    v_list.append(grad_v.detach().cpu().numpy().reshape(-1))
V = np.concatenate(v_list)

rho = 1e-3 * mw / V 
df_density_gas_keep['rho_pred'] = rho
# 检查是否与rho_GEoS相等
flag = False
for i in range(len(df_density_gas_keep)):
    if abs(df_density_gas_keep.loc[i, 'rho_pred'] - df_density_gas_keep.loc[i, 'rho_GEoS']) > 0.1:
        print(f"Difference at index {i}: rho_pred = {df_density_gas_keep.loc[i, 'rho_pred']}, rho_GEoS = {df_density_gas_keep.loc[i, 'rho_GEoS']}")
        flag = True
if not flag:
    print("All values are equal within the tolerance.")
else:
    print("There are differences between rho_pred and rho_GEoS (Liquid).")

Loading model from ../Training/model/model_DGM_fine_1131796.pth
Loading model from ../Training/model/model_DGM_fine_18094361.pth
Loading model from ../Training/model/model_DGM_fine_26317076.pth
Loading model from ../Training/model/model_DGM_fine_26318864.pth
Loading model from ../Training/model/model_DGM_fine_157564161.pth
Loading model from ../Training/model/model_DGM_fine_218105633.pth
Loading model from ../Training/model/model_DGM_fine_448585456.pth
Loading model from ../Training/model/model_DGM_fine_539035672.pth
Loading model from ../Training/model/model_DGM_fine_609641183.pth
Loading model from ../Training/model/model_DGM_fine_1211200801.pth
All values are equal within the tolerance.
